# Projet Deep Learning for Computer Vision : Évaluation

**Auteurs :** Olivier BOROT, Marin NAGY  
**Date :** Février 2026

Ce notebook effectue l'évaluation finale du modèle entraîné sur le jeu de test (jamais vu).

## Objectifs
1.  Charger le meilleur modèle (`best_model.pt`).
2.  Charger la liste des images de test (`test_images.json`).
3.  Calculer les métriques globales (Precision, Recall, F1-Score).
4.  Visualiser les prédictions.

---
## 1. Importations et Configuration

In [32]:
import os
import json
import torch
import numpy as np

# PATCH: Patch for NumPy 2.0.0+ (yolov5.utils.metrics uses np.trapz which was removed)
# This restores compatibility with yolov5 on newer numpy versions.
if not hasattr(np, 'trapz'):
    np.trapz = np.trapezoid

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from tqdm import tqdm
from pathlib import Path
import sys

# Configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

IMG_SIZE = 640
CONF_THRES = 0.25  # Seuil de confiance pour l'inférence
IOU_THRES = 0.45   # Seuil NMS IoU

# Noms des classes (doit matcher l'entraînement)
names = ['ferme', 'immeuble', 'maison', 'piscine', 'usine', 'villa']
print(f"Classes: {names}")


Device: cpu
Classes: ['ferme', 'immeuble', 'maison', 'piscine', 'usine', 'villa']



## 2. Chargement du Modèle et des Données
Chargement du modèle `best_model.pt` et du fichier `test_images.json` générés par le notebook d'entraînement.


In [33]:
# Chargement du modèle
model_path = 'yolov5n_custom.pt'
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Le modèle {model_path} est introuvable. Avez-vous exécuté Final_01_Training.ipynb ?")

print(f"Chargement du modèle depuis {model_path}...")
# On charge le checkpoint complet. 
# weights_only=False est nécessaire car nous chargeons un modèle complet (pas seulement les poids)
# et nous faisons confiance à la source (notre propre entraînement).
ckpt = torch.load(model_path, map_location=DEVICE, weights_only=False)
model = ckpt['model'].float().to(DEVICE).eval()
print("Modèle chargé avec succès.")

# Chargement du jeu de test
test_json_path = 'test_images.json'
if os.path.exists(test_json_path):
    with open(test_json_path, 'r') as f:
        test_images = json.load(f)
    print(f"Jeu de test chargé : {len(test_images)} images.")
else:
    print("ATTENTION : 'test_images.json' introuvable. Utilisation d'un dossier par défaut ou arrêt.")
    test_images = []

# Utilitaires YOLOv5 (nécessaires pour l'évaluation)
from yolov5.utils.general import non_max_suppression, scale_boxes
from yolov5.utils.metrics import box_iou, ap_per_class

Chargement du modèle depuis yolov5n_custom.pt...
Modèle chargé avec succès.
Jeu de test chargé : 15 images.


## 3. Évaluation Quantitative
Nous calculons la Précision, le Rappel et le F1-Score pour chaque classe.

In [34]:
def process_batch(detections, labels, iouv):
    """
    Compare les détections aux labels pour calculer les TP (True Positives).
    """
    correct = torch.zeros(detections.shape[0], len(iouv), dtype=torch.bool, device=iouv.device)
    iou = box_iou(labels[:, 1:], detections[:, :4])
    x = torch.where((iou >= iouv[0]) & (labels[:, 0:1] == detections[:, 5]))  # IoU > seuil AND same class
    
    if x[0].shape[0]:
        matches = torch.cat((torch.stack(x, 1), iou[x[0], x[1]][:, None]), 1).cpu().numpy()
        if x[0].shape[0] > 1:
            matches = matches[matches[:, 2].argsort()[::-1]]
            matches = matches[np.unique(matches[:, 1], return_index=True)[1]]
            matches = matches[matches[:, 2].argsort()[::-1]]
            matches = matches[np.unique(matches[:, 0], return_index=True)[1]]
        
        matches = torch.from_numpy(matches).to(iouv.device)
        correct[matches[:, 1].long()] = matches[:, 2:3] >= iouv
        
    return correct

stats = []
iouv = torch.tensor([0.5], device=DEVICE) # IoU threshold pour l'évaluation (mAP@0.5)

print("Lancement de l'évaluation...")

for img_path in tqdm(test_images):
    # --- Préparation Image ---
    try:
        img0 = Image.open(img_path).convert('RGB')
    except:
        continue
        
    w0, h0 = img0.size
    img = img0.resize((IMG_SIZE, IMG_SIZE))
    img_tensor = torch.from_numpy(np.array(img)).permute(2, 0, 1).float().to(DEVICE) / 255.0
    img_tensor = img_tensor.unsqueeze(0) # Batch dim

    # --- Préparation Targets (Labels) ---
    label_path = img_path.replace('images', 'labels').replace('.jpg', '.txt').replace('.png', '.txt')
    targets = []
    if os.path.exists(label_path) and os.path.getsize(label_path) > 0:
        # Load format: class x y w h (normalized)
        t = np.loadtxt(label_path).reshape(-1, 5)
        # Convert to pixel coordinates [class, x1, y1, x2, y2]
        # x, y, w, h are normalized center_x, center_y, width, height
        # box conversion: xywh (center) -> xyxy (corners)
        labels = torch.from_numpy(t).to(DEVICE)
        
        # Denormalize to IMG_SIZE
        # x_c, y_c, w, h
        x_c = labels[:, 1] * IMG_SIZE
        y_c = labels[:, 2] * IMG_SIZE
        w   = labels[:, 3] * IMG_SIZE
        h   = labels[:, 4] * IMG_SIZE
        
        x1 = x_c - w/2
        y1 = y_c - h/2
        x2 = x_c + w/2
        y2 = y_c + h/2
        
        # New format: [class, x1, y1, x2, y2]
        targets = torch.stack((labels[:, 0], x1, y1, x2, y2), 1)
    else:
        targets = torch.zeros((0, 5), device=DEVICE)

    # --- Inférence ---
    with torch.no_grad():
        preds = model(img_tensor)[0]
        # NMS
        preds = non_max_suppression(preds, CONF_THRES, IOU_THRES)
    
    # --- Statistiques ---
    for i, pred in enumerate(preds):
        npred = len(pred)
        ncls = len(names)
        tcls = targets[:, 0].tolist() if len(targets) else []
        
        if npred == 0:
            if len(tcls):
                stats.append((torch.zeros(0, iouv.numel(), dtype=torch.bool), torch.Tensor(), torch.Tensor(), tcls))
            continue
        
        # Preds are in IMG_SIZE coordinates (640x640), targets too.
        # Format pred: x1, y1, x2, y2, conf, class
        
        if len(targets):
            correct = process_batch(pred, targets, iouv)
        else:
            correct = torch.zeros(npred, iouv.numel(), dtype=torch.bool)
            
        stats.append((correct.cpu(), pred[:, 4].cpu(), pred[:, 5].cpu(), tcls))

# --- Calcul des Métriques Globales ---
stats = [np.concatenate(x, 0) for x in zip(*stats)]
if len(stats) and stats[0].any():
    # ap_per_class expects 'names' to be a dictionary, not a list.
    names_dict = dict(enumerate(names))
    tp, fp, p, r, f1, ap, ap_class = ap_per_class(*stats, plot=False, names=names_dict)
    ap50 = ap[:, 0]
    
    print(f"\n{'Class':<15} {'Images':<10} {'Labels':<10} {'P':<10} {'R':<10} {'F1':<10} {'mAP@.5':<10}")
    for i, c in enumerate(ap_class):
        print(f"{names[c]:<15} {len(test_images):<10} {sum(stats[3]==c):<10} {p[i]:.3f}      {r[i]:.3f}      {f1[i]:.3f}      {ap50[i]:.3f}")
    
    print(f"\nGlobal mAP@0.5: {ap50.mean():.3f}")
else:
    print("Aucune détection ou aucun label trouvé.")


Lancement de l'évaluation...


  0%|          | 0/15 [00:00<?, ?it/s]

100%|██████████| 15/15 [00:01<00:00, 12.65it/s]


Class           Images     Labels     P          R          F1         mAP@.5    
ferme           15         6          0.000      0.000      0.000      0.000
immeuble        15         174        0.000      0.000      0.000      0.000
maison          15         160        0.425      0.212      0.283      0.267
piscine         15         31         0.245      0.161      0.194      0.152
usine           15         8          0.000      0.000      0.000      0.000
villa           15         3          0.000      0.000      0.000      0.000

Global mAP@0.5: 0.070


## 4. Résultats Qualitatifs (Visualisation)

Affichons les prédictions sur quelques images du jeu de test pour visualiser les boîtes englobantes trouvées par le modèle (en **Rouge**) par rapport aux vérités terrains (en **Vert**).

In [35]:
def plot_image_with_boxes(img_path, model, conf_thres=0.25):
    # Charger image
    img0 = Image.open(img_path).convert('RGB')
    w0, h0 = img0.size
    
    # Inference
    img = img0.resize((IMG_SIZE, IMG_SIZE))
    img_tensor = torch.from_numpy(np.array(img)).permute(2, 0, 1).float().to(DEVICE) / 255.0
    img_tensor = img_tensor.unsqueeze(0)
    
    with torch.no_grad():
        preds = model(img_tensor)[0]
        preds = non_max_suppression(preds, conf_thres, IOU_THRES)
        
    # Plot
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    ax.imshow(img0)
    ax.axis('off')
    
    # 1. Dessiner Ground Truth (Vert)
    label_path = img_path.replace('images', 'labels').replace('.jpg', '.txt').replace('.png', '.txt')
    if os.path.exists(label_path) and os.path.getsize(label_path) > 0:
        l = np.loadtxt(label_path).reshape(-1, 5)
        # Norm xywh -> pixel xyxy
        w, h = w0, h0
        for box in l:
             cls, x_c, y_c, bw, bh = box
             x1 = (x_c - bw/2) * w
             y1 = (y_c - bh/2) * h
             rect = patches.Rectangle((x1, y1), bw*w, bh*h, linewidth=2, edgecolor='g', facecolor='none', linestyle='--')
             ax.add_patch(rect)
             ax.text(x1, y1-5, f"{names[int(cls)]} (GT)", color='g', fontsize=9, fontweight='bold')

    # 2. Dessiner Prédictions (Rouge)
    for pred in preds:
        if len(pred):
            # Scale boxes back to original image size
            pred[:, :4] = scale_boxes(img_tensor.shape[2:], pred[:, :4], img0.size).round()
            
            for *xyxy, conf, cls in pred:
                x1, y1, x2, y2 = xyxy
                w_box = x2 - x1
                h_box = y2 - y1
                rect = patches.Rectangle((x1, y1), w_box, h_box, linewidth=2, edgecolor='r', facecolor='none')
                ax.add_patch(rect)
                ax.text(x1, y1-5, f"{names[int(cls)]} {conf:.2f}", color='r', fontsize=9, fontweight='bold')
                
    plt.title(f"Image: {os.path.basename(img_path)}")
    plt.show()

# Afficher 3 exemples aléatoires
for i in range(3):
    img_path = np.random.choice(test_images)
    plot_image_with_boxes(img_path, model)